In [ ]:
!pip install pm4py

In [ ]:
import os
import pandas as pd
import pm4py  # version 2.7.4
from pprint import pprint  # pretty printing
from pm4py.algo.discovery.alpha import algorithm as alpha_miner
from pm4py.algo.discovery.heuristics import algorithm as heuristics_miner
from pm4py.algo.discovery.inductive import algorithm as inductive_miner
from pm4py.algo.evaluation import algorithm as evaluation
from pm4py.objects.conversion.log import converter as stream_converter
from pm4py.objects.log.importer.xes import importer as xes_import
from pm4py.visualization.petri_net import visualizer as pn_visualizer
from pm4py.algo.conformance.tokenreplay import algorithm as token_replay
import pulp

In [ ]:
# Load CSV
df = pd.read_csv("caseid_day_HypoHyper/processed/event_log.csv")



display(df.head(5))

# Rename columns to pm4py names
df = df.rename(columns={
    "case_id": "case:concept:name",
    "event_type": "concept:name",
    "timestamp": "time:timestamp",
    "value": "glucose_value"  
})

# Convert columns string
df["case:concept:name"] = df["case:concept:name"].astype(str)
df["concept:name"] = df["concept:name"].astype(str)

# Convert timestamp to datetime
df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

# Order per caso and timestamp 
df = df.sort_values(["case:concept:name", "time:timestamp"])

# Convert to EventLog
event_log = pm4py.convert_to_event_log(df)


log_df = df.copy()

print("Event log loaded successfully!")
print(f"Total de traces (casos): {len(event_log)}")
print(f"Total de eventos: {len(log_df)}")

In [ ]:

print("trace (case) atributes:")
print(list(event_log[0].attributes.keys()))

print("\nevent fields:")
print(list(event_log[0][0].keys()))


print("\nunique activities:")
print(log_df["concept:name"].unique().tolist())


print("\nstart activities:")
print(pm4py.get_start_activities(event_log))

print("\nend activities:")
print(pm4py.get_end_activities(event_log))

In [ ]:
# Heuristics Miner 
h_net, h_im, h_fm = heuristics_miner.apply(event_log)

# Inductive Miner
tree = inductive_miner.apply(event_log)
i_net, i_im, i_fm = pm4py.convert_to_petri_net(tree)

# --- Inductive Miner (Infrequent) ---
tree = inductive_miner.apply(
    event_log,
    variant=inductive_miner.Variants.IMd,
    parameters={
        "noise_threshold": 0.2 
    }
)

inf_net, inf_im, inf_fm = pm4py.convert_to_petri_net(tree)

# Alpha Miner
a_net, a_im, a_fm = alpha_miner.apply(event_log)


# 1. ILP Miner
ilp_net, ilp_im, ilp_fm = pm4py.discover_petri_net_ilp(
    event_log,
    activity_key='concept:name',
    case_id_key='case:concept:name',
    timestamp_key='time:timestamp')


print("Models generated successfuly!")

In [ ]:


#!apt-get update >/dev/null 2>&1
#!apt-get install -y graphviz >/dev/null 2>&1



In [ ]:
from pm4py.visualization.petri_net import visualizer as pn_visualizer


In [ ]:

# ===============================
# models visualisation
# ===============================

# --- Heuristics Miner ---
h_gviz = pn_visualizer.apply(h_net, h_im, h_fm)
pn_visualizer.view(h_gviz)
pn_visualizer.save(h_gviz, "caseid_day_HypoHyper/graphs/heuristics_miner_petrinet.png")

# --- Inductive Miner ---
i_gviz = pn_visualizer.apply(i_net, i_im, i_fm)
pn_visualizer.view(i_gviz)
pn_visualizer.save(i_gviz, "caseid_day_HypoHyper/graphs/inductive_miner_petrinet.png")

# --- Alpha Miner ---
a_gviz = pn_visualizer.apply(a_net, a_im, a_fm)
pn_visualizer.view(a_gviz)
pn_visualizer.save(a_gviz, "caseid_day_HypoHyper/graphs/alpha_miner_petrinet.png")

# --- ILP Miner ---
ilp_gviz = pn_visualizer.apply(ilp_net, ilp_im, ilp_fm)
pn_visualizer.view(ilp_gviz)
pn_visualizer.save(ilp_gviz, "caseid_day_HypoHyper/graphs/ilp_miner_petrinet.png")

print("Visualizations created and saved!")


In [ ]:
## --- Recall and Comprehensibility ---


def compute_recall(eval_result):
    precision = eval_result["precision"]
    fscore = eval_result["fscore"]


    if precision == 0 or (2 * precision - fscore) == 0:
        return 0.0

    recall = (fscore * precision) / (2 * precision - fscore)
    return recall

def compute_comprehensibility(net):
    num_places = len(net.places)
    num_transitions = len(net.transitions)

    total_elements = num_places + num_transitions

    if total_elements == 0:
        return 0.0

    # lower structure, bigger score
    return 1 / total_elements



In [ ]:
# Evaluate Heuristics Miner
eval_h = evaluation.apply(event_log, h_net, h_im, h_fm)
recall_h = compute_recall(eval_h)
eval_h["recall"] = recall_h
compreh_h = compute_comprehensibility(h_net)
eval_h["comprehensibility"] = compreh_h

# Evaluate Inductive Miner
eval_i = evaluation.apply(event_log, i_net, i_im, i_fm)
recall_i = compute_recall(eval_i)
eval_i["recall"] = recall_i
compreh_i = compute_comprehensibility(i_net)
eval_i["comprehensibility"] = compreh_i

# Evaluate Alpha Miner
eval_a = evaluation.apply(event_log, a_net, a_im, a_fm)
recall_a = compute_recall(eval_a)
eval_a["recall"] = recall_a
compreh_a = compute_comprehensibility(a_net)
eval_a["comprehensibility"] = compreh_a

# Evaluate ILP Miner
eval_ilp = evaluation.apply(event_log, ilp_net, ilp_im, ilp_fm)
recall_ilp = compute_recall(eval_ilp)
eval_ilp["recall"] = recall_ilp
compreh_ilp = compute_comprehensibility(ilp_net)
eval_ilp["comprehensibility"] = compreh_ilp

print("Evaluation - Heuristics Miner:")
pprint(eval_h)

print("\nEvaluation - Inductive Miner:")
pprint(eval_i)

print("\nEvaluation - Alpha Miner:")
pprint(eval_a)

print("\nEvaluation - ILP Miner:")
pprint(eval_ilp)



import os
import json

output_path = "caseid_day_HypoHyper/graphs/miner_evaluation.txt"

with open(output_path, "w") as f:
    f.write("=====================================\n")
    f.write("     Evaluation of PROCESS MINERS\n")
    f.write("=====================================\n\n")

    
    def write_block(name, data):
        f.write(f"### {name} ###\n")
        
        
        for metric in ["comprehensibility", "fscore", "precision", "recall", 
                       "generalization", "simplicity", "metricsAverageWeight"]:
            if metric in data:
                f.write(f"{metric:25s}: {data[metric]}\n")

        f.write("\n-- Fitness --\n")
        f.write(json.dumps(data.get("fitness", {}), indent=4))
        f.write("\n\n")


    write_block("Heuristics Miner", eval_h)
    write_block("Inductive Miner", eval_i)
    write_block("Alpha Miner", eval_a)
    write_block("ILP Miner", eval_ilp)

print("File created in:", output_path)



In [ ]:
# Which traces fit perfectly no modelo (Heuristics)
print("\n Replaying log with TBR - Heuristics Miner:")
replayed_h = token_replay.apply(event_log, h_net, h_im, h_fm)

# Which traces do not fit
not_fit_h = sum(1 for trace in replayed_h if not trace['trace_is_fit'])
total_h = len(replayed_h)

print(f"Traces which do not adjust to the model: {not_fit_h} de {total_h}")
print(f"Compliance rate: {(1 - not_fit_h/total_h)*100:.1f}%")

print("\n Replaying log with TBR - Inductive Miner:")
replayed_i = token_replay.apply(event_log, i_net, i_im, i_fm)

# Which traces do not fit
not_fit_i = sum(1 for trace in replayed_i if not trace['trace_is_fit'])
total_i = len(replayed_i)

print(f"Traces which do not adjust to the model: {not_fit_i} de {total_i}")
print(f"Compliance rate: {(1 - not_fit_i/total_i)*100:.1f}%")

print("\n Replaying log with TBR - Alpha Miner:")
replayed_a = token_replay.apply(event_log, a_net, a_im, a_fm)

# Which traces do not fit
not_fit_a = sum(1 for trace in replayed_a if not trace['trace_is_fit'])
total_a = len(replayed_a)

print(f"Traces which do not adjust to the model: {not_fit_a} de {total_a}")
print(f"Compliance rate: {(1 - not_fit_a/total_a)*100:.1f}%")

print("\n Replaying log with TBR - ILP Miner:")
replayed_ilp = token_replay.apply(event_log, ilp_net, ilp_im, ilp_fm)

# Which traces do not fit
not_fit_ilp = sum(1 for trace in replayed_ilp if not trace['trace_is_fit'])
total_ilp = len(replayed_ilp)

print(f"Traces which do not adjust to the model: {not_fit_ilp} de {total_ilp}")
print(f"Compliance rate: {(1 - not_fit_ilp/total_ilp)*100:.1f}%")

In [ ]:
# Avg time between "Meal" and "PPGR_peak"
meal_to_peak = log_df[log_df["concept:name"] == "PPGR_peak"].copy()
meal_to_peak = meal_to_peak.merge(
    log_df[log_df["concept:name"] == "Meal"][["case:concept:name", "time:timestamp"]].rename(
        columns={"time:timestamp": "meal_time"}
    ),
    on="case:concept:name",
    how="inner"
)

meal_to_peak["time_to_peak"] = (
    meal_to_peak["time:timestamp"] - meal_to_peak["meal_time"]
).dt.total_seconds() / 60  

print(f"\nAvg time to reach the peak: {meal_to_peak['time_to_peak'].mean():.1f} minutes")

# Avg value of peak
peak_values = log_df[log_df["concept:name"] == "PPGR_peak"]["glucose_value"]
print(f"Peak avg value: {peak_values.mean():.2f} mmol/L")